# Structured Output

- Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

# Pydantic 
- Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.6-27b")

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(decription="This is the title of the movie")
    year:int=Field(decription="This is the year which it got released")
    director:str=Field(decription="This is the director of the movied")
    rating:float=Field(decription="The movie rating out of 10")


/var/folders/s_/2xfqdh_j1pl0bdbycvv8y7n40000gn/T/ipykernel_36443/440469695.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'decription'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  title:str= Field(decription="This is the title of the movie")
/var/folders/s_/2xfqdh_j1pl0bdbycvv8y7n40000gn/T/ipykernel_36443/440469695.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'decription'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  year:int=Field(decription="This is the year which it got released")
/var/folders/s_/2xfqdh_j1pl0bdbycvv8y7n40000gn/T/ipykernel_36443/440469695.py:6: PydanticDeprecatedSince20: Using ex

In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.0', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x11558a630>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x117bd1eb0>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'decription': 'This is the title of the movie', 'type': 'string'}, 'year': {'decription': 'This is the year which it got released', 'type': 'integer'}, 'director': {'decription': 'This is the director of the movied', 'type': 'string'}, 'rating': {'decription': 'The movie rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_call

In [5]:
response = model.invoke("Provide all details about the movie Inception")
response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Request:** The user wants "all details about the movie Inception". This is a broad request, so I need to provide a comprehensive yet organized overview covering all key aspects of the film.\n\n2.  **Identify Key Categories for Movie Details:**\n   - Basic Information (Title, Release Year, Director, Writers, Studio, Runtime, Budget, Box Office)\n   - Plot Summary (Brief, without major spoilers if possible, but detailed enough)\n   - Cast & Characters\n   - Production & Development\n   - Themes & Concepts\n   - Music & Soundtrack\n   - Reception & Awards\n   - Cultural Impact & Legacy\n   - Fun Facts/Trivia\n   - Technical Aspects (Cinematography, VFX, Editing)\n   - Sequels/Related Works\n\n3.  **Gather Accurate Information (Mental Knowledge + Verification Strategy):**\n   I\'ll rely on my training data, which includes well-documented facts about Inception (2010). I\'ll structure it logically and ensure accuracy.\n\n   *Ba

In [6]:
response = model_with_structure.invoke("Provide all details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [7]:
model_with_structure = model.with_structured_output(Movie, include_raw=True)


response = model_with_structure.invoke("Provide all details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking for details about the movie "Inception".\nI have a tool named "Movie" which requires "title", "year", "director", and "rating" as parameters.\nI need to retrieve these details for "Inception".\nTitle: Inception\nYear: 2010\nDirector: Christopher Nolan\nRating: 8.8 (Commonly known rating on IMDB)\n\nI have all the required information to make the tool call.\n\nParameters:\ntitle: "Inception"\nyear: 2010\ndirector: "Christopher Nolan"\nrating: 8.8\n\nI will call the Movie function with these parameters.\n', 'tool_calls': [{'id': 'bq2qq5aer', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 213, 'prompt_tokens': 359, 'total_tokens': 572, 'completion_time': 0.405844343, 'completion_tokens_details': {'reasoning_tokens': 143}, 'prompt_time': 0.0263944

In [9]:
#Nested Structure

from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title:str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide all details about the movie Inception")

response


MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal Cobb'), Actor(name='Michael Caine', role='Professor Miles')], genres=['Science Fiction', 'Action', 'Thriller', 'Mystery'], budget=160.0)

## TypedDict

In [10]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_withtypedict=model.with_structured_output(MovieDict)

response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [11]:
from typing_extensions import TypedDict, Annotated
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: Annotated[float | None, "Budget in millions USD" ]

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'The Avengers',
 'year': 2012}

In [ ]:
# Typedict is used mainly in static Analysis
# Pydantic is API, LLMs, FastAPI
#Typedict is super fast
# Pydantic slightly slower

In [ ]:
#Middleware